In [15]:
#importing libraries
# site scrapping - https://jiji.ng/api_web/v1/listing?slug=cars&page=1&webp=True
import re
import requests
import pandas as pd

In [16]:
def extract_make_model_year(title):
    pattern = r"(?P<make>\b[a-zA-Z\-]+)\s+(?P<model>[A-Za-z0-9\-]+(?:\s[A-Za-z0-9\-]+)?)?.*?(?P<year>\d{4})(?!\d)"
    match = re.search(pattern, title)
    if match:
        return match.group("make"), match.group("model"), match.group("year")
    return None, None, None

In [17]:
def extract_condition(condition):
    condition_lower = condition.lower()
    if "foreign used" in condition_lower:
        return "foreign used"
    elif "local used" in condition_lower:
        return "local used"
    elif "brand new" in condition_lower:
        return "brand new"
    else:
        return None

In [18]:
def extract_transmission(transmission):
    transmission_lower = transmission.lower()
    if "automatic" in transmission_lower:
        return "automatic"
    elif "manual" in transmission_lower:
        return "manual"
    else:
        return None

In [22]:
#communicatting with API
def fetch_data(page):
    url = "https://jiji.ng/api_web/v1/listing"
    params = {
        "slug": "cars",
        "page": page,
        "webp": True
    }

    headers = {
        "User_Agent": "Mozilla/5.0"
    }
    try:
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
    except requests.RequestException:
        print(f"[Page] {page}, Request error")
        return []
    except ValueError:
        print(f"[Page] {page}, Decode error")

    adverts = data.get("adverts_list", {}).get("adverts", [])
    if not isinstance(adverts, list):
        print(f"A error occured, Expected list but got {type(adverts)}")
        return []

    return adverts

In [26]:
def get_attr_value(attrs, key_name):
    for attr in attrs:
        if attr.get("name", "").lower() == key_name.lower():
            return attr.get("value", "").strip()
    return None

In [27]:
#control function - carries all functionality of the code
def main():
    all_ads = []
    for page in range(1, 101):
        ads = fetch_data(page)
        print(f"Page {page}. {len(ads)} ads found.")

        for ad in ads:
            if isinstance(ad, dict):
                attrs = ad.get("attrs", [])
                title = ad.get("title", "")
                condition_ = get_attr_value(attrs, "condition")
                transmission_ = get_attr_value(attrs, "transmission")
                condition = extract_condition(condition_)
                transmission = extract_transmission(transmission_)
                make, model, year = extract_make_model_year(title)
                price = ad.get("price_title", "")

                if price:
                    all_ads.append({
                        "title": title,
                        "make": make,
                        "model": model,
                        "year": year,
                        "condition": condition, 
                        "transmission": transmission,
                        "price": price
                    })
    if all_ads:
        df = pd.DataFrame(all_ads)
        df.to_csv("jiji_car_dataset.csv", index=False)
        print("Successfully extracted car dataset from Jiji.")
    else:
        print("Ops! Nothing to extract.")

main()

Page 1. 20 ads found.
Page 2. 20 ads found.
Page 3. 20 ads found.
Page 4. 20 ads found.
Page 5. 20 ads found.
Page 6. 20 ads found.
Page 7. 20 ads found.
Page 8. 20 ads found.
Page 9. 20 ads found.
Page 10. 20 ads found.
Page 11. 20 ads found.
Page 12. 20 ads found.
Page 13. 20 ads found.
Page 14. 20 ads found.
Page 15. 20 ads found.
Page 16. 20 ads found.
Page 17. 20 ads found.
Page 18. 20 ads found.
Page 19. 20 ads found.
Page 20. 20 ads found.
Page 21. 20 ads found.
Page 22. 20 ads found.
Page 23. 20 ads found.
Page 24. 20 ads found.
Page 25. 20 ads found.
Page 26. 20 ads found.
Page 27. 20 ads found.
Page 28. 20 ads found.
Page 29. 20 ads found.
Page 30. 20 ads found.
Page 31. 20 ads found.
Page 32. 20 ads found.
Page 33. 20 ads found.
Page 34. 20 ads found.
Page 35. 20 ads found.
Page 36. 20 ads found.
Page 37. 20 ads found.
Page 38. 20 ads found.
Page 39. 20 ads found.
Page 40. 20 ads found.
Page 41. 20 ads found.
Page 42. 20 ads found.
Page 43. 20 ads found.
Page 44. 20 ads foun

In [33]:
df = pd.read_csv("jiji_car_dataset.csv")
df.isnull().sum()

title           0
make            7
model           8
year            7
condition       0
transmission    4
price           0
dtype: int64

In [34]:
df.head()

,title,make,model,year,condition,transmission,price
0,Mercedes-Benz GLC-Class 2018 Gray,Mercedes-Benz,GLC-Class,2018.0,local used,automatic,"₦ 17,500,000"
1,Honda Accord 2021 Black,Honda,Accord,2021.0,foreign used,automatic,"₦ 34,000,000"
2,Kia Optima EX 4dr Sedan (2.4L 4cyl 6A) 2014 Red,Kia,Optima EX,2014.0,foreign used,automatic,"₦ 13,650,000"
3,Toyota Camry 2012 White,Toyota,Camry,2012.0,local used,automatic,"₦ 9,500,000"
4,Lexus RX 2006 Gold,Lexus,RX,2006.0,local used,automatic,"₦ 7,700,000"
